# Somatic mutations and mutational signatures in cancer


Today, we are going to explore somatic mutations in different cancer types and learn about mutational signatures. To this end, data of 3 cancer types: Breast cancer (BRCA), Lung adenocarcinoma (LUAD) and Skin cancer melanoma (SKCM) have been prepared as csv files and uploaded to OLAT. The data were generated by randomly sampling 100 patients from each cancer type in the TCGA dataset. The data for today's class was generated using the R package sigminer and the class was inspired by the tutorial: https://shixiangwang.github.io/sigminer-book/basic-workflow.html

This tutorial assumes that you have uploaded the csv files to Google drive


In [ ]:
# import the modules/packages that we will be using during this class
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive') # this will prompt you to grant Colab access your data on your Google Drive

# Define paths (Adjust 'MyDrive/Biomedical_Informatics/data/' if your folder structure differs)
base_path = '/content/drive/MyDrive/Biomedical_Informatics/data/'
cancer_types = ['BRCA', 'LUAD', 'SKCM']

def load_cancer_data(ctype):
    return {
        'mutations': pd.read_csv(f"{base_path}{ctype}_mutations.csv"),
        'clinical': pd.read_csv(f"{base_path}{ctype}_clinical.csv"),
        'summary': pd.read_csv(f"{base_path}{ctype}_gene_summary.csv"),
        'tally': pd.read_csv(f"{base_path}{ctype}_tally.csv"),
        'signatures': pd.read_csv(f"{base_path}{ctype}_signatures.csv",
                                  index_col=0)
    }

# Load all data into a nested dictionary
data = {ctype: load_cancer_data(ctype) for ctype in cancer_types}
print("Data loaded successfully for BRCA, LUAD, and SKCM.")

In [ ]:
# Formatting for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


First, let's look at the mutational data

In [ ]:
print("--- Mutation Data (MAF-like) ---")
display(data['BRCA']['mutations'].head())

In [ ]:
# Calculate total mutation counts per cancer type
total_muts = {ctype: len(data[ctype]['mutations']) for ctype in cancer_types}
mut_df = pd.DataFrame(list(total_muts.items()), columns=['Cancer Type', 'Total Mutations'])

plt.figure(figsize=(8, 5))
sns.barplot(data=mut_df, x='Cancer Type', hue='Cancer Type', legend=False,
            y='Total Mutations', palette='viridis')
plt.title('Total Somatic Mutation Count by Cancer Type')
plt.ylabel('Number of Mutations')
plt.show()

Let's explore the clinical data

In [ ]:
print("--- Clinical Data ---")
#pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', 20)
display(data['BRCA']['clinical'].head())

In [ ]:
data['BRCA']['clinical'].columns

In [ ]:
for ctype in cancer_types:
    df_clin = data[ctype]['clinical']

    print(f"\n{'='*10} {ctype} Clinical Metadata {'='*10}")
    print(f"Columns: {df_clin.columns.tolist()}")

    # Check for missing values
    missing = df_clin.isnull().sum()
    print("\nMissing values per column:")
    print(missing[missing > 0])

In [ ]:
for ctype in cancer_types:
    df_clin = data[ctype]['clinical']

    # 3. Plot Proportions of Race, Gender, and Histological Type
    cols_to_plot = ['CDR_race', 'CDR_gender', 'CDR_histological_type']
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    for i, col in enumerate(cols_to_plot):
        if col in df_clin.columns:
            # Calculate percentages
            counts = df_clin[col].value_counts(normalize=True) * 100
            sns.barplot(x=counts.index, hue=counts.index, y=counts.values,
                        legend=False, ax=axes[i], palette='Set2')
            axes[i].set_title(f'Proportion of {col}\n({ctype})')
            axes[i].set_ylabel('Percentage (%)')
            axes[i].tick_params(axis='x', rotation=45)
        else:
            axes[i].set_title(f'{col} Not Found')
            axes[i].text(0.5, 0.5, 'Column Missing', ha='center')

    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(6, 3))

age_data = []

for ctype in cancer_types:
    df_clin = data[ctype]['clinical']

    # Identify the age column (handling potential naming variations)
    age_col = [c for c in df_clin.columns if 'age' in c.lower()][0]

    # Clean data (remove NAs and ensure numeric)
    clean_age = pd.to_numeric(df_clin[age_col], errors='coerce').dropna()

    # Prepare data for Boxplot comparison
    temp_df = pd.DataFrame({'Age': clean_age, 'Cancer': ctype})
    age_data.append(temp_df)

combined_age = pd.concat(age_data)
sns.boxplot(data=combined_age, x='Cancer', hue='Cancer',
            legend=False, y='Age', palette='Set3')

plt.tight_layout()
plt.show()

# Print summary statistics
for ctype in cancer_types:
    df_clin = data[ctype]['clinical']
    age_col = [c for c in df_clin.columns if 'age' in c.lower()][0]
    median_age = pd.to_numeric(df_clin[age_col], errors='coerce').median()
    print(f"Median age for {ctype}: {median_age:.1f} years")

In [ ]:
for ctype in cancer_types:
    df_clin = data[ctype]['clinical']

    # Plot Clinical Stage distribution
    stage_col = [c for c in df_clin.columns if 'stage' in c.lower()]
    if stage_col:
        plt.figure(figsize=(6, 3))
        sns.countplot(data=df_clin, x=stage_col[0],
                      hue=stage_col[0], legend=False,
                      palette='viridis',
                      order=sorted(df_clin[stage_col[0]].dropna().unique()))
        plt.title(f'Patient Distribution by Stage: {ctype}')
        plt.xticks(rotation=45)
        plt.show()

Which genes are have the most mutations?

In [ ]:
print("\n--- Gene Summary Data ---")
display(data['BRCA']['summary'].head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharey=False)

for i, ctype in enumerate(cancer_types):
    # Get top 10 mutated genes from the summary file
    top_genes = data[ctype]['summary'].sort_values(by='total', ascending=False).head(10)

    sns.barplot(ax=axes[i], data=top_genes, x='total', y='Hugo_Symbol',
                hue='Hugo_Symbol', legend=False, palette='magma')
    axes[i].set_title(f'Top mutated genes in {ctype}')
    axes[i].set_xlabel('Number of mutations')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for i, ctype in enumerate(cancer_types):
    # Use 'MutatedSamples' column as requested
    top_genes = data[ctype]['summary'].sort_values(by='MutatedSamples', ascending=False).head(15)

    sns.barplot(ax=axes[i], data=top_genes, x='MutatedSamples', y='Hugo_Symbol',
                hue='Hugo_Symbol', legend=False, palette='flare')
    axes[i].set_title(f'Top Driver Genes in {ctype}\n(by Number of Samples)')
    axes[i].set_xlabel('Number of Mutated Samples')

plt.tight_layout()
plt.show()

Plotting single base substitutions with their genomic contexts

In [ ]:
print("\n--- SBS Tally Data (96 contexts) ---")
display(data['BRCA']['tally'].head())

In [ ]:
# Color mapping for the 6 substitution types
colors_map = {
    'C>A': 'lightblue',
    'C>G': 'black',
    'C>T': 'red',
    'T>A': 'grey',
    'T>C': 'lightgreen',
    'T>G': 'pink'
}

for ctype in cancer_types:
    df_tally = data[ctype]['tally']
    # 1. Identify SBS columns (assumes patient ID is the first column)
    sbs_cols = list(df_tally.columns[1:])
    # 2. Sort columns by the mutation type
    sbs_cols_sorted = sorted(sbs_cols, key=lambda x: next((sub for sub in colors_map if sub in x), x))
    # 3. Calculate mean counts across patients for the sorted columns
    profile = df_tally[sbs_cols_sorted].mean()
    # 4. Map colors to the sorted sequence
    bar_colors = []
    for col_name in profile.index:
        match = next((sub for sub in colors_map if sub in col_name), None)
        bar_colors.append(colors_map.get(match, 'silver'))

    plt.figure(figsize=(18, 5))
    profile.plot(kind='bar', color=bar_colors, width=0.8)
    plt.title(f'Mean SBS-96 Mutational Profile (Sorted): {ctype}')
    plt.ylabel('Mean Mutation Count')
    plt.xlabel('Trinucleotide Context (Sorted by Substitution)')
    plt.xticks(fontsize=7, rotation=90)
    plt.show()

Investigate Cosmic signatures

In [ ]:
print("\n--- Signature Weights ---")
display(data['BRCA']['signatures'].head())

In [ ]:
sig_comparison = pd.DataFrame()

for ctype in cancer_types:
    df_sig = data[ctype]['signatures']

    # Calculate the mean weight for each signature across all samples (columns)
    sig_comparison[ctype] = df_sig.mean(axis=1)

fig, axes = plt.subplots(3, 1, figsize=(14, 15), sharex=True)

for i, ctype in enumerate(cancer_types):
    # Plot the raw mean weights for each signature
    sns.barplot(ax=axes[i], x=sig_comparison.index, hue= sig_comparison.index,
                y=sig_comparison[ctype], legend=False, palette='viridis')
    axes[i].set_title(f'Absolute Mean COSMIC Signature Weights: {ctype}')
    axes[i].set_ylabel('Mean Weight')
    axes[i].tick_params(axis='x', rotation=90)

plt.xlabel('COSMIC Signature ID')
plt.tight_layout()
plt.show()

In [ ]:
sig_comparison = pd.DataFrame()

for ctype in cancer_types:
    df_sig = data[ctype]['signatures']
    sig_comparison[ctype] = df_sig.mean(axis=1)
# Visualization with improved scaling
# We use 'z_score=0' to normalize by row (Signatures)
# This shows which cancer type "owns" that signature relative to the others
plt.figure(figsize=(10, 12))
sns.clustermap(sig_comparison,
               annot=True,
               cmap='RdBu_r',
               z_score=0, # Standardize across the row (across cancer types)
               cbar_kws={'label': 'Z-score'},
               col_cluster=False) # Keep the cancer types in our specific order

plt.title('Relative COSMIC Signature Weights (Row-Normalized)')
plt.show()

# Bonus exercises

Correlation between age and COSMIC_1 signature

In [ ]:
correlation_results = []

for ctype in cancer_types:
    # 1. Process Signature Data
    df_sig = data[ctype]['signatures']

    # Extract COSMIC_1 and transpose so samples are rows
    sig1_weights = df_sig.loc[['COSMIC_1']].T
    sig1_weights.columns = ['Signature_1_Weight']

    # TRUNCATE: Chop Sample IDs to 12 characters to match clinical data
    sig1_weights.index = sig1_weights.index.str[:12]
    sig1_weights.index.name = 'Sample_ID'

    # 2. Process Clinical Data
    df_clin = data[ctype]['clinical']
    id_col = df_clin.columns[0]
    age_col = 'CDR_age_at_initial_pathologic_diagnosis'

    # Ensure IDs are 12 chars and set as index
    clin_subset = df_clin[[id_col, age_col]].copy()
    clin_subset.columns = ['Sample_ID', 'Age']
    clin_subset['Sample_ID'] = clin_subset['Sample_ID'].str[:12]
    clin_subset = clin_subset.set_index('Sample_ID')

    # 3. Merge
    merged_data = sig1_weights.join(clin_subset, how='inner')
    merged_data['Cancer Type'] = ctype
    correlation_results.append(merged_data)

# Combine datasets
final_corr_df = pd.concat(correlation_results)
final_corr_df['Age'] = pd.to_numeric(final_corr_df['Age'], errors='coerce')
final_corr_df = final_corr_df.dropna(subset=['Age', 'Signature_1_Weight'])

# Create the Scatterplot with Per-Type Trendlines
plt.figure(figsize=(12, 8))

# Define colors from Set3 for consistency
palette_colors = sns.color_palette("Set3", len(cancer_types))
color_map = dict(zip(cancer_types, palette_colors))

# Plot scatter points and trendlines for each cancer type
for ctype in cancer_types:
    subset = final_corr_df[final_corr_df['Cancer Type'] == ctype]

    # Plot the points
    sns.scatterplot(
        data=subset, x='Age',  y='Signature_1_Weight',
        color=color_map[ctype], s=70, alpha=0.6, edgecolor='gray', label=ctype
    )

    # Plot the individual regression line (trendline)
    sns.regplot(
        data=subset, x='Age', y='Signature_1_Weight',
        scatter=False, color=color_map[ctype], label=f'{ctype} Trend'
    )

plt.title('Biological Clock: Age vs. Signature 1 Weight (Per Cancer Type)', fontsize=16)
plt.xlabel('Age at Initial Pathologic Diagnosis', fontsize=12)
plt.ylabel('Signature 1 Absolute Weight', fontsize=12)
plt.legend(title='Cancer Type & Trend', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Calculate statistics per type
for ctype in cancer_types:
    subset = final_corr_df[final_corr_df['Cancer Type'] == ctype]
    r_val = subset['Age'].corr(subset['Signature_1_Weight'])
    print(f"Correlation for {ctype}: R = {r_val:.4f}")

PCA based on mutational signatures

In [ ]:
#We will standardize the signature weights so that each signature contributes
#equally to the analysis, then project the samples onto the first two principal
#components.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Prepare the signature matrix for all samples
pca_data_list = []

for ctype in cancer_types:
    # Signatures are rows, Samples are columns
    df_sig = data[ctype]['signatures']

    # Transpose so each Row is a Sample and each Column is a Signature weight
    df_transposed = df_sig.T
    df_transposed['Cancer Type'] = ctype
    pca_data_list.append(df_transposed)

# Combine into one master dataframe
combined_sig_df = pd.concat(pca_data_list)

# Separate the features (signature weights) from the target (cancer type)
features = combined_sig_df.drop(columns=['Cancer Type'])
labels = combined_sig_df['Cancer Type']

# 2. Standardize the features
# PCA is sensitive to variance, so we scale weights to have Mean=0 and Variance=1
# $z = \frac{x - \mu}{\sigma}$
x_scaled = StandardScaler().fit_transform(features)

# 3. Run PCA
# We reduce the 30 COSMIC signatures down to 2 Principal Components (PCs)
pca = PCA(n_components=2)
pcs = pca.fit_transform(x_scaled)

# Create a dataframe for the results
pca_df = pd.DataFrame(data=pcs, columns=['PC1', 'PC2'])
pca_df['Cancer Type'] = labels.values

# 4. Visualize the results
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=pca_df,
    x='PC1',
    y='PC2',
    hue='Cancer Type',
    palette='Set3',
    s=80,
    alpha=0.8,
    edgecolor='black'
)

# Annotate the plot with the amount of variance explained by each PC
explained_variance = pca.explained_variance_ratio_
plt.xlabel(f'Principal Component 1 ({explained_variance[0]:.1%} Variance)')
plt.ylabel(f'Principal Component 2 ({explained_variance[1]:.1%} Variance)')
plt.title('PCA of Cancer Samples: Separation by Mutational Signatures', fontsize=16)
plt.legend(title='Cancer Type', loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Print the top contributors to PC1
# This tells us which signatures are driving the separation
loadings = pd.DataFrame(pca.components_[0], index=features.columns, columns=['PC1_Loading'])
top_sigs = loadings.abs().sort_values(by='PC1_Loading', ascending=False).head(5)
print("\nTop Signatures driving PC1 separation:")
print(top_sigs)
loadings2 = pd.DataFrame(pca.components_[1], index=features.columns, columns=['PC2_Loading'])
top_sigs2 = loadings2.abs().sort_values(by='PC2_Loading', ascending=False).head(5)
print("\nTop Signatures driving PC2 separation:")
print(top_sigs2)